In [1]:
import tiktoken
from openai import OpenAI
import os
import ast

import pandas as pd
import numpy as np

# 嵌入模型
openai_base_url = os.getenv("OPENAI_BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(
    base_url=openai_base_url,
    api_key=openai_api_key,
)
embedding_model = 'Qwen/Qwen3-Embedding-8B'


In [2]:
df = pd.read_csv('output/embedding_1000.csv')

df['embedding_vec'] = df['embedding'].apply(ast.literal_eval)
df.head(1)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,count_token,embedding,embedding_vec
0,0,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title:where does one start...and stop... with...,53,"[-0.014720369130373001, 0.035861968994140625, ...","[-0.014720369130373001, 0.035861968994140625, ..."


In [3]:
# 在向量空间中，语义相似的词或文本单位。距离靠近

def cosine_distance(vec1, vec2):
    """
    计算两个向量之间的余弦距离
    :param vec1: 向量1
    :param vec2: 向量2
    :return: 余弦距离
    """
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def embedding_text(text, model):
    """
    通过Embedding模型处理文本数据
    :param text: 需要处理的文本
    :param model: 嵌入模型
    :return: 文本向量
    """
    resp = client.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding


def search_by_keyword(df, keyword, top_n=10, print_flag=True):
    """
    根据关键词搜索最相似的文本
    :param print_flag: 是否打印结果
    :param df: 数据框
    :param keyword: 关键词
    :param top_n: 返回最相似的n个文本
    :return: 最相似的n个文本
    """
    word_embedding = embedding_text(keyword, model=embedding_model)
    # 计算相识度
    df['similarity'] = df['embedding_vec'].apply(lambda x: cosine_distance(x, word_embedding))
    result = (
        df.sort_values(by='similarity', ascending=False)
        .head(top_n)['combined'].str.replace('Title:', '')
        .str.replace('; Content:', ';')
    )
    if print_flag:
        for i in result :
            print(i)

    return result

In [12]:
search_by_keyword(df, 'Grandma Chili', top_n=3)

Grandma's Chili Powder Replica Very Good;This product is what Grandma's Chili Powder used to be. People have posted and searched online for the no longer available Grandma's Chili Powder. Well this Williams Chili Season Mix is the same thing. It is great. Try this.
Great for HS lunch;Great for HS lunch, kid enjoy as a snack also, will buy again. Salted chips are good too, tried them too.
Great for HS lunch;Great for HS lunch, kid enjoy as a snack also, will buy again. Salted chips are good too, tried them too.


782    Grandma's Chili Powder Replica Very Good;This ...
577    Great for HS lunch;Great for HS lunch, kid enj...
881    Great for HS lunch;Great for HS lunch, kid enj...
Name: combined, dtype: str